# Entraînement sur GPU (Google Colab)

Entraîne chaque config de `CONFIGS`, l'évalue sur toute la validation (20 pulses par configuration, 13 SNR) et copie
chaque dossier de run sur Google Drive.

**Avant de lancer**, le dossier `DRIVE_DIR` de Google Drive doit contenir :

- `CODE_ZIP` : le code du dépôt, avec les configs à entraîner ;
- `recorded_dataset.zip` : les données d'entraînement et de validation (2,5 Go).

Si le dossier est partagé avec toi, ajoute-le d'abord à ton Drive : `Partagés avec moi`, clic droit,
`Organiser > Ajouter un raccourci > Mon Drive`.

Puis : `Exécution > Modifier le type d'exécution > GPU T4`, et `Exécution > Tout exécuter`.

**Règle** : seules les données d'entraînement et de validation vont sur Drive et Colab. Le blind test n'y va
jamais.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/Linkiw/radarml"   # contient CODE_ZIP et recorded_dataset.zip
CODE_ZIP = "radarml_code_seed43.zip"
CONFIGS = ["configs/e005_seed43.yaml", "configs/e006_seed43.yaml"]

## 1. Vérifier le GPU

In [ ]:
import torch

!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.is_available(), "Pas de GPU : Exécution > Modifier le type d'exécution > GPU T4, puis relance"
print("PyTorch", torch.__version__, "| GPU :", torch.cuda.get_device_name(0))

## 2. Connecter Google Drive et installer le code

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
for name in (CODE_ZIP, "recorded_dataset.zip"):
    assert os.path.isfile(f"{DRIVE_DIR}/{name}"), f"Fichier manquant sur Drive : {DRIVE_DIR}/{name}"

!rm -rf /content/repo && mkdir -p /content/repo
!unzip -q "{DRIVE_DIR}/{CODE_ZIP}" -d /content/repo
%cd /content/repo
!pip install --quiet -r requirements.txt && pip install --quiet -e .

## 3. Préparer les données (environ 5 min)

In [ ]:
# Copie sur le disque de Colab : lire 660 fichiers directement sur Drive ralentirait l'entraînement.
!cp "{DRIVE_DIR}/recorded_dataset.zip" /content/recorded_dataset.zip
!python scripts/setup_data.py --zip /content/recorded_dataset.zip --skip-file-hashes
!python -m pytest -q

## 4. Entraîner puis évaluer chaque config

Les configs de `CONFIGS` passent l'une après l'autre. Pour chacune : une ligne par époque, une évaluation rapide
(`SCORE VALIDATION ...`), une copie sur Drive, l'évaluation complète, puis une seconde copie sur Drive. Si la
session est coupée, les runs déjà copiés sont sur Drive : retire leurs configs de `CONFIGS` et relance.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

import yaml

for config in CONFIGS:
    experiment = yaml.safe_load(Path(config).read_text(encoding="utf-8"))["experiment"]
    print("=" * 30, experiment, "=" * 30, flush=True)
    subprocess.run([sys.executable, "scripts/train.py", "--config", config, "--device", "cuda"], check=True)
    run = sorted(Path("runs").glob(f"{experiment}_*"))[-1]
    drive_run = Path(DRIVE_DIR) / "runs" / run.name
    shutil.copytree(run, drive_run, dirs_exist_ok=True)
    print("Entraînement copié sur Drive :", drive_run, flush=True)
    subprocess.run([sys.executable, "scripts/evaluate.py", "--run", str(run), "--device", "cuda"], check=True)
    shutil.copytree(run, drive_run, dirs_exist_ok=True)
    print("Copié sur Drive :", drive_run, flush=True)
print("TOUT EST TERMINÉ")

## 6. Terminé

Télécharge les dossiers `runs/<experiment>_...` de `DRIVE_DIR` et place-les dans le dossier `runs\` du dépôt sur
ton PC. Les temps d'inférence de ces runs sont des temps GPU : ils ne se comparent pas aux runs CPU du journal.